In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import ndimage
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'stixsans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

base_output_dir = r'PATH_TO_BASE_OUTPUT_DIR'
base_data_path = r'PATH_TO_BASE_DATA_DIR'
mask_path = r'PATH_TO_MASK_FILE'

time_scale_ranges = ['0-1', '2-3', '4-6', '7-12', '13-18', '19-24', '25-48']
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
start_years = range(2001, 2012)
time_windows = [(year, year + 9) for year in start_years]

timescale_colors = {
    '0-1': {
        'light': '#E6F3FF',
        'dark': '#0476E7'
    },
    '2-3': {
        'light': '#E0F7FA',
        'dark': '#00b2cb'
    },
    '4-6': {
        'light': '#E8F5E9',
        'dark': '#00AA09'
    },
    '7-12': {
        'light': '#F8EEC2',
        'dark': '#FFD000'
    },
    '13-18': {
        'light': '#F7D2BA',
        'dark': '#FF6600'
    },
    '19-24': {
        'light': '#B9BAD8',
        'dark': '#3F45F8'
    },
    '25-48': {
        'light': '#F5D0D5',
        'dark': '#F10726'
    }
}

combined_output_dir = os.path.join(base_output_dir, 'combined_output')
os.makedirs(combined_output_dir, exist_ok=True)

with rasterio.open(mask_path) as mask_src:
    mask_data = mask_src.read(1)
    mask_transform = mask_src.transform
    mask_crs = mask_src.crs

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

def interpolate_color(light_hex, dark_hex, value):
    light_rgb = hex_to_rgb(light_hex)
    dark_rgb = hex_to_rgb(dark_hex)

    r = light_rgb[0] + (dark_rgb[0] - light_rgb[0]) * value
    g = light_rgb[1] + (dark_rgb[1] - light_rgb[1]) * value
    b = light_rgb[2] + (dark_rgb[2] - light_rgb[2]) * value

    return (r, g, b)

for season_idx, season in enumerate(seasons, 1):
    timescale_frequencies = {}
    reference_lat = None
    reference_lon = None
    mask_binary = None

    for ts_idx, time_scale_range in enumerate(time_scale_ranges, 1):
        accumulated_overlap = None
        mask_initialized = False
        processed_windows = 0

        for start_year, end_year in time_windows:
            file_name = f'max_corr_{start_year}-{end_year}_{season}.nc'
            file_path = os.path.join(base_data_path, file_name)

            if not os.path.exists(file_path):
                continue

            try:
                ds = xr.open_dataset(file_path)
                time_scale_original = ds['time_scale'].values.copy()

                try:
                    lon = ds['lon'].values
                    lat_original = ds['lat'].values.copy()
                except:
                    try:
                        lon = ds['longitude'].values
                        lat_original = ds['latitude'].values.copy()
                    except:
                        ds.close()
                        continue

                ds.close()

                if not mask_initialized:
                    height, width = time_scale_original.shape
                    lon_min, lon_max = lon.min(), lon.max()
                    lat_min, lat_max = lat_original.min(), lat_original.max()

                    dst_transform = rasterio.transform.from_bounds(
                        lon_min, lat_min, lon_max, lat_max, width, height
                    )

                    mask_resampled = np.zeros((height, width), dtype=np.float32)

                    reproject(
                        source=mask_data,
                        destination=mask_resampled,
                        src_transform=mask_transform,
                        src_crs=mask_crs,
                        dst_transform=dst_transform,
                        dst_crs='EPSG:4326',
                        resampling=Resampling.nearest
                    )

                    mask_binary_test = (mask_resampled > 0).astype(bool)
                    if np.sum(mask_binary_test) > 0:
                        valid_rows = np.where(np.any(mask_binary_test, axis=1))[0]
                        if len(valid_rows) > 0:
                            center_row = height // 2
                            upper_half = np.sum(mask_binary_test[:center_row, :])
                            lower_half = np.sum(mask_binary_test[center_row:, :])

                            need_flip = False
                            first_lat = lat_original[valid_rows[0]]

                            if lat_original[0] > lat_original[-1]:
                                if lower_half > upper_half * 2 and first_lat < 0:
                                    need_flip = True
                            else:
                                if upper_half > lower_half * 2 and first_lat < 0:
                                    need_flip = True

                            if need_flip:
                                mask_resampled = np.flipud(mask_resampled)

                    mask_binary = (mask_resampled > 0).astype(bool)
                    accumulated_overlap = np.zeros(time_scale_original.shape, dtype=int)
                    reference_lat = lat_original
                    reference_lon = lon
                    mask_initialized = True

                time_scale_masked = np.where(mask_binary, time_scale_original, np.nan)

                if time_scale_range == '0-1':
                    range_mask = (time_scale_masked >= 0) & (time_scale_masked <= 1)
                elif time_scale_range == '2-3':
                    range_mask = (time_scale_masked >= 2) & (time_scale_masked <= 3)
                elif time_scale_range == '4-6':
                    range_mask = (time_scale_masked >= 4) & (time_scale_masked <= 6)
                elif time_scale_range == '7-12':
                    range_mask = (time_scale_masked >= 7) & (time_scale_masked <= 12)
                elif time_scale_range == '13-18':
                    range_mask = (time_scale_masked >= 13) & (time_scale_masked <= 18)
                elif time_scale_range == '19-24':
                    range_mask = (time_scale_masked >= 19) & (time_scale_masked <= 24)
                elif time_scale_range == '25-48':
                    range_mask = (time_scale_masked >= 25) & (time_scale_masked <= 48)

                binary_mask = (range_mask & mask_binary).astype(np.uint8)
                labeled_array, num_features = ndimage.label(binary_mask, structure=np.ones((3, 3)))
                unique_labels, counts = np.unique(labeled_array, return_counts=True)
                large_clusters = unique_labels[(counts > 100) & (unique_labels > 0)]
                focal_mask = np.isin(labeled_array, large_clusters)
                accumulated_overlap += focal_mask.astype(int)
                processed_windows += 1

            except Exception:
                continue

        if accumulated_overlap is not None and np.max(accumulated_overlap) > 0:
            timescale_frequencies[time_scale_range] = {
                'frequency': accumulated_overlap.copy(),
                'max_freq': np.max(accumulated_overlap),
                'total_pixels': np.sum(accumulated_overlap > 0),
                'processed_windows': processed_windows
            }

    if len(timescale_frequencies) > 0 and reference_lat is not None:
        lat = reference_lat.copy()
        if lat[0] < lat[-1]:
            lat = lat[::-1]
            display_flipped = True
        else:
            display_flipped = False

        fig_height = 4.0
        aspect_ratio = 2.2
        fig_width = fig_height * aspect_ratio

        fig, ax = plt.subplots(
            1,
            1,
            figsize=(fig_width, fig_height),
            subplot_kw={'projection': ccrs.PlateCarree()}
        )

        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='black', zorder=10)
        ax.add_feature(cfeature.BORDERS, linewidth=0.2, color='gray', zorder=10)
        ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.3, zorder=0)
        ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.2, zorder=0)

        for time_scale_range in reversed(time_scale_ranges):
            if time_scale_range not in timescale_frequencies:
                continue

            frequency_data = timescale_frequencies[time_scale_range]['frequency']
            max_freq = timescale_frequencies[time_scale_range]['max_freq']

            if display_flipped:
                frequency_display = np.flipud(frequency_data)
            else:
                frequency_display = frequency_data

            frequency_normalized = frequency_display / max_freq
            light_color = timescale_colors[time_scale_range]['light']
            dark_color = timescale_colors[time_scale_range]['dark']

            height, width = frequency_display.shape
            rgba_array = np.zeros((height, width, 4))

            for i in range(height):
                for j in range(width):
                    if frequency_display[i, j] > 0:
                        freq_ratio = frequency_normalized[i, j]
                        r, g, b = interpolate_color(light_color, dark_color, freq_ratio)
                        rgba_array[i, j, 0] = r
                        rgba_array[i, j, 1] = g
                        rgba_array[i, j, 2] = b
                        rgba_array[i, j, 3] = 0.8
                    else:
                        rgba_array[i, j, 3] = 0

            ax.imshow(
                rgba_array,
                extent=[reference_lon.min(), reference_lon.max(), lat.min(), lat.max()],
                transform=ccrs.PlateCarree(),
                origin='upper',
                zorder=5
            )

        season_short_labels = {
            'Spring': 'MAM',
            'Summer': 'JJA',
            'Autumn': 'SON',
            'Winter': 'DJF'
        }

        ax.set_title(
            f'{season_short_labels[season]}',
            fontsize=20,
            pad=8,
            fontweight='normal'
        )

        gl = ax.gridlines(
            draw_labels=True,
            dms=True,
            x_inline=False,
            y_inline=False,
            linewidth=0.3,
            alpha=0.5,
            linestyle='--'
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 10, 'weight': 'normal'}
        gl.ylabel_style = {'size': 10, 'weight': 'normal'}

        from matplotlib.patches import Rectangle

        legend_x = 0.015
        legend_y = 0.015
        legend_width = 0.18
        legend_height = 0.48

        legend_bg = Rectangle(
            (legend_x, legend_y),
            legend_width,
            legend_height,
            transform=ax.transAxes,
            facecolor='white',
            edgecolor='#2C2C2C',
            linewidth=0.8,
            alpha=0.98,
            zorder=20
        )
        ax.add_patch(legend_bg)

        ax.text(
            legend_x + legend_width / 2,
            legend_y + legend_height - 0.015,
            ' ',
            transform=ax.transAxes,
            fontsize=12,
            fontweight='bold',
            ha='center',
            va='top',
            color='#2C2C2C',
            zorder=21
        )

        bar_height = 0.045
        bar_spacing = 0.012
        start_y = legend_y + legend_height - 0.08

        bar_width = 0.04
        bar_x = legend_x + 0.012

        freq_column_width = 0.035
        freq_column_right = legend_x + legend_width - 0.012
        freq_column_center_x = freq_column_right - freq_column_width / 2

        for idx, time_scale_range in enumerate(time_scale_ranges):
            if time_scale_range not in timescale_frequencies:
                continue

            bar_y = start_y - idx * (bar_height + bar_spacing)

            light_color = timescale_colors[time_scale_range]['light']
            dark_color = timescale_colors[time_scale_range]['dark']

            n_segments = 20
            segment_width = bar_width / n_segments

            for seg_idx in range(n_segments):
                seg_x = bar_x + seg_idx * segment_width
                ratio = seg_idx / (n_segments - 1)
                r, g, b = interpolate_color(light_color, dark_color, ratio)

                seg_rect = Rectangle(
                    (seg_x, bar_y),
                    segment_width,
                    bar_height,
                    transform=ax.transAxes,
                    facecolor=(r, g, b),
                    edgecolor='none',
                    zorder=21
                )
                ax.add_patch(seg_rect)

            border_rect = Rectangle(
                (bar_x, bar_y),
                bar_width,
                bar_height,
                transform=ax.transAxes,
                facecolor='none',
                edgecolor='#2C2C2C',
                linewidth=0.5,
                zorder=22
            )
            ax.add_patch(border_rect)

            label_text = f'{time_scale_range}'
            text_x = bar_x + bar_width + 0.008
            text_y = bar_y + bar_height / 2

            ax.text(
                text_x,
                text_y,
                label_text,
                transform=ax.transAxes,
                fontsize=12,
                fontweight='normal',
                va='center',
                ha='left',
                color='#2C2C2C',
                zorder=23
            )

            max_freq = timescale_frequencies[time_scale_range]['max_freq']
            freq_text = f'{max_freq}'

            ax.text(
                freq_column_center_x,
                text_y,
                freq_text,
                transform=ax.transAxes,
                fontsize=12,
                fontweight='normal',
                va='center',
                ha='center',
                color='#555555',
                family='monospace',
                zorder=23
            )

        ax.text(
            legend_x + legend_width / 2,
            legend_y + 0.008,
            'Max frequency',
            transform=ax.transAxes,
            fontsize=10,
            style='italic',
            ha='center',
            va='bottom',
            color='#666666',
            zorder=21
        )

        plt.tight_layout()

        output_filename = f'nature_style_gradient_{season}.png'
        output_path = os.path.join(combined_output_dir, output_filename)
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')

        plt.show()

        stats_data = []
        for time_scale_range in time_scale_ranges:
            if time_scale_range in timescale_frequencies:
                info = timescale_frequencies[time_scale_range]
                stats_data.append({
                    'timescale': time_scale_range,
                    'light_color': timescale_colors[time_scale_range]['light'],
                    'dark_color': timescale_colors[time_scale_range]['dark'],
                    'total_pixels': info['total_pixels'],
                    'max_frequency': info['max_freq'],
                    'processed_windows': info['processed_windows'],
                    'avg_frequency': np.mean(info['frequency'][info['frequency'] > 0])
                })

        if stats_data:
            df_stats = pd.DataFrame(stats_data)
            csv_filename = f'nature_style_statistics_{season}.csv'
            csv_path = os.path.join(combined_output_dir, csv_filename)
            df_stats.to_csv(csv_path, index=False, encoding='utf-8-sig')

    else:
        pass


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import ndimage
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'stixsans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

base_output_dir = r'PATH_TO_BASE_OUTPUT_DIR'
base_data_path = r'PATH_TO_BASE_DATA_DIR'
mask_path = r'PATH_TO_MASK_FILE'

time_scale_ranges = ['0-1', '2-3', '4-6', '7-12', '13-18', '19-24', '25-48']
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
start_years = range(2001, 2012)
time_windows = [(year, year + 9) for year in start_years]

FREQUENCY_THRESHOLD = 5

timescale_colors = {
    '0-1': {
        'light': '#E6F3FF',
        'dark': '#0476E7'
    },
    '2-3': {
        'light': '#E0F7FA',
        'dark': '#00b2cb'
    },
    '4-6': {
        'light': '#E8F5E9',
        'dark': '#00AA09'
    },
    '7-12': {
        'light': '#F8EEC2',
        'dark': '#FFD000'
    },
    '13-18': {
        'light': '#F7D2BA',
        'dark': '#FF6600'
    },
    '19-24': {
        'light': '#B9BAD8',
        'dark': '#3F45F8'
    },
    '25-48': {
        'light': '#F5D0D5',
        'dark': '#F10726'
    }
}

combined_output_dir = os.path.join(base_output_dir, f'combined_output_freq_gt_{FREQUENCY_THRESHOLD}')
os.makedirs(combined_output_dir, exist_ok=True)

with rasterio.open(mask_path) as mask_src:
    mask_data = mask_src.read(1)
    mask_transform = mask_src.transform
    mask_crs = mask_src.crs

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

def interpolate_color(light_hex, dark_hex, value):
    light_rgb = hex_to_rgb(light_hex)
    dark_rgb = hex_to_rgb(dark_hex)

    r = light_rgb[0] + (dark_rgb[0] - light_rgb[0]) * value
    g = light_rgb[1] + (dark_rgb[1] - light_rgb[1]) * value
    b = light_rgb[2] + (dark_rgb[2] - light_rgb[2]) * value

    return (r, g, b)

for season_idx, season in enumerate(seasons, 1):
    timescale_frequencies = {}
    reference_lat = None
    reference_lon = None
    mask_binary = None

    for ts_idx, time_scale_range in enumerate(time_scale_ranges, 1):
        accumulated_overlap = None
        mask_initialized = False
        processed_windows = 0

        for start_year, end_year in time_windows:
            file_name = f'max_corr_{start_year}-{end_year}_{season}.nc'
            file_path = os.path.join(base_data_path, file_name)

            if not os.path.exists(file_path):
                continue

            try:
                ds = xr.open_dataset(file_path)
                time_scale_original = ds['time_scale'].values.copy()

                try:
                    lon = ds['lon'].values
                    lat_original = ds['lat'].values.copy()
                except:
                    try:
                        lon = ds['longitude'].values
                        lat_original = ds['latitude'].values.copy()
                    except:
                        ds.close()
                        continue

                ds.close()

                if not mask_initialized:
                    height, width = time_scale_original.shape
                    lon_min, lon_max = lon.min(), lon.max()
                    lat_min, lat_max = lat_original.min(), lat_original.max()

                    dst_transform = rasterio.transform.from_bounds(
                        lon_min, lat_min, lon_max, lat_max, width, height
                    )

                    mask_resampled = np.zeros((height, width), dtype=np.float32)

                    reproject(
                        source=mask_data,
                        destination=mask_resampled,
                        src_transform=mask_transform,
                        src_crs=mask_crs,
                        dst_transform=dst_transform,
                        dst_crs='EPSG:4326',
                        resampling=Resampling.nearest
                    )

                    mask_binary_test = (mask_resampled > 0).astype(bool)
                    if np.sum(mask_binary_test) > 0:
                        valid_rows = np.where(np.any(mask_binary_test, axis=1))[0]
                        if len(valid_rows) > 0:
                            center_row = height // 2
                            upper_half = np.sum(mask_binary_test[:center_row, :])
                            lower_half = np.sum(mask_binary_test[center_row:, :])

                            need_flip = False
                            first_lat = lat_original[valid_rows[0]]

                            if lat_original[0] > lat_original[-1]:
                                if lower_half > upper_half * 2 and first_lat < 0:
                                    need_flip = True
                            else:
                                if upper_half > lower_half * 2 and first_lat < 0:
                                    need_flip = True

                            if need_flip:
                                mask_resampled = np.flipud(mask_resampled)

                    mask_binary = (mask_resampled > 0).astype(bool)
                    accumulated_overlap = np.zeros(time_scale_original.shape, dtype=int)
                    reference_lat = lat_original
                    reference_lon = lon
                    mask_initialized = True

                time_scale_masked = np.where(mask_binary, time_scale_original, np.nan)

                if time_scale_range == '0-1':
                    range_mask = (time_scale_masked >= 0) & (time_scale_masked <= 1)
                elif time_scale_range == '2-3':
                    range_mask = (time_scale_masked >= 2) & (time_scale_masked <= 3)
                elif time_scale_range == '4-6':
                    range_mask = (time_scale_masked >= 4) & (time_scale_masked <= 6)
                elif time_scale_range == '7-12':
                    range_mask = (time_scale_masked >= 7) & (time_scale_masked <= 12)
                elif time_scale_range == '13-18':
                    range_mask = (time_scale_masked >= 13) & (time_scale_masked <= 18)
                elif time_scale_range == '19-24':
                    range_mask = (time_scale_masked >= 19) & (time_scale_masked <= 24)
                elif time_scale_range == '25-48':
                    range_mask = (time_scale_masked >= 25) & (time_scale_masked <= 48)

                binary_mask = (range_mask & mask_binary).astype(np.uint8)
                labeled_array, num_features = ndimage.label(binary_mask, structure=np.ones((3, 3)))
                unique_labels, counts = np.unique(labeled_array, return_counts=True)
                large_clusters = unique_labels[(counts > 100) & (unique_labels > 0)]
                focal_mask = np.isin(labeled_array, large_clusters)
                accumulated_overlap += focal_mask.astype(int)
                processed_windows += 1

            except Exception:
                continue

        if accumulated_overlap is not None and np.max(accumulated_overlap) > 0:
            timescale_frequencies[time_scale_range] = {
                'frequency': accumulated_overlap.copy(),
                'max_freq': np.max(accumulated_overlap),
                'total_pixels': np.sum(accumulated_overlap > 0),
                'processed_windows': processed_windows
            }

    if len(timescale_frequencies) > 0 and reference_lat is not None:
        lat = reference_lat.copy()
        if lat[0] < lat[-1]:
            lat = lat[::-1]
            display_flipped = True
        else:
            display_flipped = False

        fig_height = 4.0
        aspect_ratio = 2.2
        fig_width = fig_height * aspect_ratio

        fig, ax = plt.subplots(
            1,
            1,
            figsize=(fig_width, fig_height),
            subplot_kw={'projection': ccrs.PlateCarree()}
        )

        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='black', zorder=10)
        ax.add_feature(cfeature.BORDERS, linewidth=0.2, color='gray', zorder=10)
        ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.3, zorder=0)
        ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.2, zorder=0)

        for time_scale_range in reversed(time_scale_ranges):
            if time_scale_range not in timescale_frequencies:
                continue

            frequency_data = timescale_frequencies[time_scale_range]['frequency']
            max_freq = timescale_frequencies[time_scale_range]['max_freq']

            if display_flipped:
                frequency_display = np.flipud(frequency_data)
            else:
                frequency_display = frequency_data

            frequency_display_filtered = np.where(
                frequency_display > FREQUENCY_THRESHOLD,
                frequency_display,
                0
            )

            valid_freq = frequency_display_filtered[frequency_display_filtered > 0]
            if len(valid_freq) == 0:
                continue

            max_freq_filtered = np.max(frequency_display_filtered)
            frequency_normalized = frequency_display_filtered / max_freq_filtered

            light_color = timescale_colors[time_scale_range]['light']
            dark_color = timescale_colors[time_scale_range]['dark']

            height, width = frequency_display_filtered.shape
            rgba_array = np.zeros((height, width, 4))

            for i in range(height):
                for j in range(width):
                    if frequency_display_filtered[i, j] > 0:
                        freq_ratio = frequency_normalized[i, j]
                        r, g, b = interpolate_color(light_color, dark_color, freq_ratio)
                        rgba_array[i, j, 0] = r
                        rgba_array[i, j, 1] = g
                        rgba_array[i, j, 2] = b
                        rgba_array[i, j, 3] = 0.8
                    else:
                        rgba_array[i, j, 3] = 0

            ax.imshow(
                rgba_array,
                extent=[reference_lon.min(), reference_lon.max(), lat.min(), lat.max()],
                transform=ccrs.PlateCarree(),
                origin='upper',
                zorder=5
            )

        season_short_labels = {
            'Spring': 'MAM',
            'Summer': 'JJA',
            'Autumn': 'SON',
            'Winter': 'DJF'
        }

        ax.set_title(
            f'{season_short_labels[season]}',
            fontsize=20,
            pad=8,
            fontweight='normal'
        )

        gl = ax.gridlines(
            draw_labels=True,
            dms=True,
            x_inline=False,
            y_inline=False,
            linewidth=0.3,
            alpha=0.5,
            linestyle='--'
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 10, 'weight': 'normal'}
        gl.ylabel_style = {'size': 10, 'weight': 'normal'}

        from matplotlib.patches import Rectangle

        legend_x = 0.015
        legend_y = 0.015
        legend_width = 0.18
        legend_height = 0.48

        legend_bg = Rectangle(
            (legend_x, legend_y),
            legend_width,
            legend_height,
            transform=ax.transAxes,
            facecolor='white',
            edgecolor='#2C2C2C',
            linewidth=0.8,
            alpha=0.98,
            zorder=20
        )
        ax.add_patch(legend_bg)

        ax.text(
            legend_x + legend_width / 2,
            legend_y + legend_height - 0.015,
            ' ',
            transform=ax.transAxes,
            fontsize=12,
            fontweight='bold',
            ha='center',
            va='top',
            color='#2C2C2C',
            zorder=21
        )

        bar_height = 0.045
        bar_spacing = 0.012
        start_y = legend_y + legend_height - 0.08

        bar_width = 0.04
        bar_x = legend_x + 0.012

        freq_column_width = 0.035
        freq_column_right = legend_x + legend_width - 0.012
        freq_column_center_x = freq_column_right - freq_column_width / 2

        for idx, time_scale_range in enumerate(time_scale_ranges):
            if time_scale_range not in timescale_frequencies:
                continue

            bar_y = start_y - idx * (bar_height + bar_spacing)

            light_color = timescale_colors[time_scale_range]['light']
            dark_color = timescale_colors[time_scale_range]['dark']

            n_segments = 20
            segment_width = bar_width / n_segments

            for seg_idx in range(n_segments):
                seg_x = bar_x + seg_idx * segment_width
                ratio = seg_idx / (n_segments - 1)
                r, g, b = interpolate_color(light_color, dark_color, ratio)

                seg_rect = Rectangle(
                    (seg_x, bar_y),
                    segment_width,
                    bar_height,
                    transform=ax.transAxes,
                    facecolor=(r, g, b),
                    edgecolor='none',
                    zorder=21
                )
                ax.add_patch(seg_rect)

            border_rect = Rectangle(
                (bar_x, bar_y),
                bar_width,
                bar_height,
                transform=ax.transAxes,
                facecolor='none',
                edgecolor='#2C2C2C',
                linewidth=0.5,
                zorder=22
            )
            ax.add_patch(border_rect)

            label_text = f'{time_scale_range}'
            text_x = bar_x + bar_width + 0.008
            text_y = bar_y + bar_height / 2

            ax.text(
                text_x,
                text_y,
                label_text,
                transform=ax.transAxes,
                fontsize=12,
                fontweight='normal',
                va='center',
                ha='left',
                color='#2C2C2C',
                zorder=23
            )

            max_freq = timescale_frequencies[time_scale_range]['max_freq']
            freq_text = f'{max_freq}'

            ax.text(
                freq_column_center_x,
                text_y,
                freq_text,
                transform=ax.transAxes,
                fontsize=12,
                fontweight='normal',
                va='center',
                ha='center',
                color='#555555',
                family='monospace',
                zorder=23
            )

        ax.text(
            legend_x + legend_width / 2,
            legend_y + 0.008,
            f'Max frequency (>{FREQUENCY_THRESHOLD})',
            transform=ax.transAxes,
            fontsize=9,
            style='italic',
            ha='center',
            va='bottom',
            color='#666666',
            zorder=21
        )

        plt.tight_layout()

        output_filename = f'nature_style_gradient_{season}_freq_gt_{FREQUENCY_THRESHOLD}.png'
        output_path = os.path.join(combined_output_dir, output_filename)
        plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')

        plt.show()

        stats_data = []
        for time_scale_range in time_scale_ranges:
            if time_scale_range in timescale_frequencies:
                info = timescale_frequencies[time_scale_range]
                freq_data = info['frequency']

                filtered_pixels = np.sum(freq_data > FREQUENCY_THRESHOLD)
                filtered_freq = freq_data[freq_data > FREQUENCY_THRESHOLD]

                stats_data.append({
                    'timescale': time_scale_range,
                    'light_color': timescale_colors[time_scale_range]['light'],
                    'dark_color': timescale_colors[time_scale_range]['dark'],
                    'total_pixels': info['total_pixels'],
                    'filtered_pixels': filtered_pixels,
                    'filter_ratio': f"{filtered_pixels / info['total_pixels'] * 100:.1f}%",
                    'max_frequency': info['max_freq'],
                    'max_frequency_filtered': np.max(filtered_freq) if len(filtered_freq) > 0 else 0,
                    'processed_windows': info['processed_windows'],
                    'avg_frequency': np.mean(info['frequency'][info['frequency'] > 0]),
                    'avg_frequency_filtered': np.mean(filtered_freq) if len(filtered_freq) > 0 else 0
                })

        if stats_data:
            df_stats = pd.DataFrame(stats_data)
            csv_filename = f'nature_style_statistics_{season}_freq_gt_{FREQUENCY_THRESHOLD}.csv'
            csv_path = os.path.join(combined_output_dir, csv_filename)
            df_stats.to_csv(csv_path, index=False, encoding='utf-8-sig')

    else:
        pass
